In [1]:
import pandas as pd
import ast
import os
import random
import M0  # Utilizing your provided M0.py file
import datetime

# 1. Configuration and Directory Setup
OUT_DIR = "../data/meal_output_data"
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

# Map the paths to your meal datasets
REQ_PATH = "../data/input_data/user_meal_requirements.csv"
MEAL_PATH = "../data/input_data/meal_categories.csv"

# 2. Data Import 
req_df = pd.read_csv(REQ_PATH)
meal_df = pd.read_csv(MEAL_PATH)

# Preprocess category strings into sets
req_df['required_categories'] = req_df['required_categories'].apply(lambda x: ast.literal_eval(x))
meal_df['categories'] = meal_df['categories'].apply(lambda x: ast.literal_eval(x))

# Create dictionaries for the M0 logic
meal_categories_dict = dict(zip(meal_df['meal_name'], meal_df['categories']))
all_meal_names = meal_df['meal_name'].tolist()

def generate_meal_m0_bundles():
    results = []
    print("Start time:\t", datetime.datetime.now())
    
    # Iterate through each user requirement row (the "proposals")
    for _, row in req_df.iterrows():
        user_id = row['user_id']
        occasion = row['meal_occasion']
        required_cats = row['required_categories']
        
        # Pick one random 'lead meal' for the bundle
        lead_meal = random.choice(all_meal_names)
        
        # Use M0.py logic to create exactly 1 bundle (team)
        # This function handles the lead meal inclusion and random sizing (1-4)
        bundles_list = M0.create_teams_for_each_person(
            all_researchers=all_meal_names, 
            target_researcher=lead_meal, 
            num_of_teams=1
        )
        
        # Extract the single bundle generated
        meal_bundle = bundles_list[0]
        
        # Use M0.py logic to calculate the goodness score (Ultra-Metric)
        # In this domain, it scores how well the random bundle covers the required categories
        goodness = M0.apply_ultra_metric(required_cats, meal_bundle, meal_categories_dict)
        
        results.append({
            'user_id': user_id,
            'meal_occasion': occasion,
            'lead_meal': lead_meal,
            'meal_bundle': meal_bundle,
            'goodness_score': goodness
        })
        
    print("End time:\t", datetime.datetime.now())
    return pd.DataFrame(results)

# 3. Execution
print("Generating 1 meal bundle per user requirement using M0 logic...")
final_results = generate_meal_m0_bundles()

# Save output
output_path = os.path.join(OUT_DIR, 'meal_results_m0_single.csv')
final_results.to_csv(output_path, index=False)

print(f"Process complete. {len(final_results)} bundles saved to {output_path}")

Generating 1 meal bundle per user requirement using M0 logic...
Start time:	 2026-02-10 17:52:56.844066
End time:	 2026-02-10 17:52:56.871590
Process complete. 500 bundles saved to ../data/meal_output_data/meal_results_m0_single.csv
